In [3]:
import pathlib
from pprint import pprint

import apoc
import numpy as np
import pyclesperanto_prototype as cle
from bioio import BioImage
from bioio.writers import OmeTiffWriter
from napari_ndev import ImageOverview, helpers
from skimage.morphology import skeletonize

c:\Users\timmo\Documents\GitHub\BastianLab\Ferritin_Phalloidin\.venv\Lib\site-packages\nyxus\nyxus.py:29: SyntaxWarning: invalid escape sequence '\p'
  """Nyxus image feature extraction library
c:\Users\timmo\Documents\GitHub\BastianLab\Ferritin_Phalloidin\.venv\Lib\site-packages\nyxus\nyxus.py:576: SyntaxWarning: invalid escape sequence '\p'
  """Sets parameters of feature GABOR
c:\Users\timmo\Documents\GitHub\BastianLab\Ferritin_Phalloidin\.venv\Lib\site-packages\nyxus\nyxus.py:693: SyntaxWarning: invalid escape sequence '\p'
  """Sets parameters of the Nyxus class
c:\Users\timmo\Documents\GitHub\BastianLab\Ferritin_Phalloidin\.venv\Lib\site-packages\nyxus\nyxus.py:757: SyntaxWarning: invalid escape sequence '\p'
  """Returns the parameters of a Nyxus object. If no args are supplied, all parameters will be returned.
c:\Users\timmo\Documents\GitHub\BastianLab\Ferritin_Phalloidin\.venv\Lib\site-packages\nyxus\nyxus.py:1832: SyntaxWarning: invalid escape sequence '\p'
  """Sets paramete

In [4]:
# Helper functions for morphology adjustments
def voronoi_label_adjustment(intensity_image, label_image):
    label_binary = cle.greater_constant(label_image, constant=0) # binarize
    intensity_th_gb = cle.gaussian_blur(intensity_image, sigma_x=1, sigma_y=1)
    intensity_peaks = cle.detect_maxima_box(intensity_th_gb, radius_x=0, radius_y=0)
    select_peaks = cle.binary_and(intensity_peaks, label_binary)
    label_voronoi = cle.masked_voronoi_labeling(select_peaks, label_binary)
    return label_voronoi

def get_pixel_class_as_objects(label, obj_class):
    obj_label = label == obj_class
    return cle.connected_components_labeling_box(obj_label)

def close_labels(label, closing_radius):
    morph_closed = cle.closing_labels(label, radius=closing_radius)
    return cle.connected_components_labeling_box(morph_closed)

def connect_breaks(label, label_connect_distance):
    label_dilated = cle.dilate_labels(label, radius=label_connect_distance/2)
    label_merged = cle.merge_touching_labels(label_dilated)
    label_connected = (label_merged * (label > 0)).astype(np.uint16)
    return label_connected

def exclude_labels(label, minimum_label_size, maximum_label_size):
    label_exclude_large = cle.exclude_labels_on_edges(label)
    label_exclude_small = cle.exclude_labels_out_of_size_range(label_exclude_large, minimum_size=minimum_label_size, maximum_size=maximum_label_size)
    return label_exclude_small

def skeletonize_labels(label):
    skeleton = skeletonize(cle.pull(label))
    skeleton_label = (label * skeleton).astype(np.uint16)
    return skeleton_label

In [5]:

working_dir = pathlib.Path('.')

raw_image_dir, raw_image_files = helpers.get_directory_and_files(
    working_dir / 'Raw_Images'
)
pprint(raw_image_files)

output_dir = working_dir / 'Processed_Images_DAPIClass'
output_dir.mkdir(exist_ok=True)

overview_dir = output_dir / 'Overviews'
overview_dir.mkdir(exist_ok=True)

data_dir = working_dir / 'Data'
data_dir.mkdir(exist_ok=True)
data_loc = data_dir / 'nyxus_all_images.csv'

phall_cl_loc = working_dir / r'Classifiers\phalloidin_4.cl'
phall_cl = apoc.ObjectSegmenter(phall_cl_loc)

ftn_cl_loc = working_dir / r'Classifiers\ferritin_1.cl'
ftn_cl = apoc.ObjectSegmenter(ftn_cl_loc)

dapi_class_cl_loc = working_dir / r'Classifiers\dapi_3_obj_class.cl'
dapi_class_cl = apoc.ObjectClassifier(dapi_class_cl_loc)


[WindowsPath('Raw_Images/2024-07-23 25x select neurons 24HIC phall AF488 ft 647 dapi obl.czi')]


In [6]:
img = BioImage(raw_image_files[0])
pixelsize = img.physical_pixel_sizes.X
print(img.channel_names)
print(img.physical_pixel_sizes)
# print(img.scenes)

['AF647', 'AF488', 'DAPI', 'Oblique']
PhysicalPixelSizes(Z=None, Y=0.1809755619211473, X=0.1809755619211473)


In [7]:
DAPI_C = 2
PHALL_C = 1
FTN_C = 0

# CONSTANTS, currently as real units, not rounded
SOMA_MINIMUM_AREA = 30 / (pixelsize * pixelsize)
# this is to exclude non-somal blobs
SOMA_MAXIMUM_AREA = 200 / (pixelsize * pixelsize)
NEURON_MINIMUM_AREA = 30 / (pixelsize * pixelsize)

# morphological operation to link breaks
INITIAL_CLOSING_DISTANCE = 0.1 / pixelsize

# for merging labels without closing based on edge-to-edge distance
LABEL_MERGE_DISTANCE = 3 / pixelsize

# a nucleus is about 10-15 microns in diameter
MAX_FERRITIN_DISTANCE = 10
MIN_NUCLEUS_SIZE = 3 # radius
MAX_NUCLEUS_SIZE = 10

# Maximum nucleus size, conservative radius size of 12.5 pixels (25um diamter)
max_ferritin_distance = MAX_FERRITIN_DISTANCE / pixelsize
min_nucleus_size = (MIN_NUCLEUS_SIZE / pixelsize) ** 2 * np.pi
max_nucleus_size = (MAX_NUCLEUS_SIZE / pixelsize) ** 2 * np.pi


# # for merging skeleton branches, the maximum value, likely best to keep this small, even if it means broken neurons
# SKELETON_MERGE_DISTANCE = 2 / pixelsize
# MINIMUM_SEGMENT_LENGTH = 2 / pixelsize

# # for selecting neurons of a minimum size, but a maximum soma size
# # (i.e. needs to be at least a soma, but not multiple)]
# # currently scaled to real units
# SHOLL_INNER_RING = 0
# SHOLL_RING_SPACING = 1
# SHOLL_OUTER_RING = 151

# # Mitochondria stuff
# MINIMUM_MITOCHONDRIA_DISTANCE = -100 / pixelsize
# MAXIMUM_MITOCHONDRIA_DISTANCE = 5 / pixelsize

# SOMA_DILATION = 3 / pixelsize


In [ ]:
nyxus_stack = []

for file in raw_image_files[:3]:
    print(file.name)
    img = BioImage(file)
    pixel_size = img.physical_pixel_sizes.X

    for scene in img.scenes[:]:
        print(scene)
        img.set_scene(scene)

        dapi = img.get_image_data("YX", C=DAPI_C)
        phall = img.get_image_data("YX", C=PHALL_C)
        ftn = img.get_image_data("YX", C=FTN_C)

        nuclei_segmented = cle.voronoi_otsu_labeling(dapi, None, 15, 2)
        nuclei_no_edges = cle.exclude_labels_on_edges(nuclei_segmented)
        nuclei_exclude = cle.exclude_labels_out_of_size_range(
            nuclei_no_edges, None, min_nucleus_size, max_nucleus_size
        )

        if np.max(nuclei_exclude) > 10:
            print('Too many nuclei, skipping')
            continue

        # DAPI classification
        dapi_class = dapi_class_cl.predict(labels=nuclei_exclude, image=dapi)
        dapi_centroids = cle.reduce_labels_to_centroids(nuclei_exclude)
        dapi_class_centroids = ((dapi_centroids > 0) * dapi_class).astype(np.uint16)

        ftn_predicted = ftn_cl.predict(ftn)
        ftn_exclude = exclude_labels(ftn_predicted, 0, 1e10)
        ftn_voronoi = voronoi_label_adjustment(ftn, ftn_exclude)
        ftn_final = ftn_voronoi

        # Phalloidin morphology processing using pixel classifier approach
        # This combines the phalloidin prediction with nuclei information
        phall_predicted = phall_cl.predict([phall, dapi])

        # Convert predicted objects to labeled objects
        phall_obj = cle.connected_components_labeling_box(phall_predicted > 0)

        # Apply morphology adjustments
        phall_closed = close_labels(phall_obj, INITIAL_CLOSING_DISTANCE)
        phall_connected = connect_breaks(phall_closed, LABEL_MERGE_DISTANCE)
        phall_exclude = exclude_labels(phall_connected, NEURON_MINIMUM_AREA, 1e10)

        phall_final = phall_exclude
        phall_skeleton = skeletonize_labels(phall_final)

        ### Create overviews and save
        ############################################
        image_dict = {
            'image': [dapi, phall, ftn],
            'title': ['DAPI', 'Phalloidin', 'Ftn'],
            'min_display_intensity': [
                np.percentile(dapi, 0.1),
                np.percentile(phall, 0.1),
                np.percentile(ftn, 0.1),
            ],
            'max_display_intensity': [
                np.percentile(dapi, 99.97),
                np.percentile(phall, 99.97),
                np.percentile(ftn, 99.97),
            ],
        }

        ### Save Labels
        label_list = [nuclei_exclude, phall_final, ftn_final, dapi_class, phall_skeleton]
        concatenated_labels = np.stack(label_list, axis=0)
        concatenated_label_names = ['DAPI', 'Phalloidin', 'Ftn', 'DAPI_Class', 'Phalloidin_Skeleton']

        label_dict = {
            'image': concatenated_labels,
            'title': concatenated_label_names,
            'labels': [True, True, True, True, True],
        }

        ImageOverview(
            image_sets=[image_dict, label_dict],
            fig_title=f'{file.stem}_{scene}'
        ).save(overview_dir, f'{file.stem}_{scene}_overview.png')

        OmeTiffWriter.save(
            data=concatenated_labels.astype(np.uint16),
            uri=output_dir / f'{file.stem}_{scene}_segmented.ome.tiff',
            physical_pixel_sizes=img.physical_pixel_sizes,
            channel_names=concatenated_label_names,
            dim_order='CYX',
        )

#         #### Measurements
#         ############################################
#         nyxus_intensity_list = [
#             ftn,
#             (ftn_predicted > 0), # binary of the label image
#         ]
#         intensity_names = ['ferritin', 'ferritin_binary']

#         nyxus_label_list = [
#             phall_final,
#             phall_final,
#         ]
#         label_names = ['neuron_label', 'neuron_label']

#         nyx = Nyxus(
#             features=[
#                 'area_um2', 'integrated_intensity', 'mean', 'median',
#             ],
#             pixels_per_micron=pixelsize,
#         )

#         nyxus_df = nyx.featurize(
#             intensity_images=np.stack(nyxus_intensity_list, axis=0),
#             label_images=np.stack(nyxus_label_list, axis=0),
#             intensity_names=intensity_names,
#             label_names=label_names,
#         )
#         nyxus_df.insert(0, 'file', file.stem)
#         nyxus_df.insert(1, 'scene', scene)
#         nyxus_stack.append(nyxus_df)

# nyxus_concat = pd.concat(nyxus_stack, ignore_index=True)
# nyxus_concat.to_csv(data_loc, index=False)

2024-07-23 25x select neurons 24HIC phall AF488 ft 647 dapi obl.czi
P9-A10
P9-A10


TypeError: ImageOverview.__init__() got an unexpected keyword argument 'image_title'